# NeuroGolf submission builder
exp_id: `GOLF_20260608_008b_jonathan_structural_pass_mix`
dataset: `octaviograu/neurogolf-manual-rewrites-v205`


In [ ]:
from pathlib import Path
import base64
import hashlib
import json
import shutil
import zipfile

EXP_ID = 'GOLF_20260608_008b_jonathan_structural_pass_mix'
GIT_COMMIT = '0b86e8b'
SOURCE_IDS = ['SRC_KAGGLE_NOTEBOOK_JONATHAN_CONSTRAINT_MIX']
DATASET_INPUT = Path('/kaggle/input/neurogolf-manual-rewrites-v205')
SOURCE_SUBDIR = 'submission'
EMBEDDED_ZIP_B64_PARTS = ['...']
PAYLOAD_FILE = Path('submission_payload.b64')
WORK = Path('/kaggle/working')
OUT_DIR = WORK / 'submission_files'
OUT_DIR.mkdir(exist_ok=True)

zip_path = WORK / 'submission.zip'
used_embedded = False
payload_b64 = ''.join(EMBEDDED_ZIP_B64_PARTS)
if not payload_b64 and PAYLOAD_FILE.name and PAYLOAD_FILE.exists():
    payload_b64 = PAYLOAD_FILE.read_text().strip()
source_dir = DATASET_INPUT / SOURCE_SUBDIR
candidate_zip = DATASET_INPUT / 'submission.zip'
if payload_b64:
    zip_path.write_bytes(base64.b64decode(payload_b64.encode('ascii')))
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
    files = [n for n in names if n.endswith('.onnx')]
    used_embedded = True
elif candidate_zip.exists():
    shutil.copy2(candidate_zip, zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
    files = [n for n in names if n.endswith('.onnx')]
    used_embedded = True
elif not source_dir.exists():
    candidates = [p for p in DATASET_INPUT.rglob('task001.onnx')]
    if candidates:
        source_dir = candidates[0].parent
    else:
        raise FileNotFoundError(f'No task001.onnx under {DATASET_INPUT}')

if not used_embedded:
    files = sorted(source_dir.glob('task*.onnx'))
    if not files:
        raise FileNotFoundError(f'No task*.onnx files under {source_dir}')

    for src in files:
        shutil.copy2(src, OUT_DIR / src.name)

    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for src in sorted(OUT_DIR.glob('task*.onnx')):
            zf.write(src, arcname=src.name)

h = hashlib.sha256()
with zip_path.open('rb') as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b''):
        h.update(chunk)

manifest = {
    'exp_id': EXP_ID,
    'git_commit': GIT_COMMIT,
    'source_ids': SOURCE_IDS,
    'dataset_slug': 'octaviograu/neurogolf-manual-rewrites-v205',
    'source_dir': 'embedded_zip_fallback' if used_embedded else str(source_dir),
    'package_sha256': h.hexdigest(),
    'file_count': len(files),
    'package_size': zip_path.stat().st_size,
}
print(json.dumps(manifest, indent=2))
print('submission.zip is ready at', zip_path)
